### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748])

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Pregunta 1

Se espera que al menos uno de los top 5 coincida con el verdadero topico. No obstante hay casos donde ninguno coincide.

In [30]:
np.random.seed(44)
selected_indices = np.random.choice(len(newsgroups_train.data), 5, replace=False)

for idx in selected_indices:
    sims = cosine_similarity(X_train[idx], X_train)[0]  # shape: (N_train,)
    top5 = np.argsort(sims)[::-1][1:6]  # exclude self at position 0
    
    query_topic = newsgroups_train.target_names[y_train[idx]]
    print(f"\n=== Query doc {idx} | Topic: {query_topic} ===")
    print(f"Text preview: {newsgroups_train.data[idx][:200].strip()}")
    print("5 most similar:")
    for rank, sim_idx in enumerate(top5, 1):
        sim_topic = newsgroups_train.target_names[y_train[sim_idx]]
        match = '✓' if sim_topic == query_topic else '✗'
        print(f"  {rank}. sim={sims[sim_idx]:.4f} | {match} {sim_topic}")


=== Query doc 868 | Topic: comp.sys.ibm.pc.hardware ===
Text preview: We are trying to install a donated hard disk (Miniscribe
vintage 1988) on a supercheap ancient Compaq XT for
use in education.  The only problem is that the
supercheap Compaq didn't come with the manu
5 most similar:
  1. sim=0.2547 | ✓ comp.sys.ibm.pc.hardware
  2. sim=0.2328 | ✓ comp.sys.ibm.pc.hardware
  3. sim=0.2204 | ✗ talk.politics.misc
  4. sim=0.2192 | ✗ comp.windows.x
  5. sim=0.2188 | ✗ sci.crypt

=== Query doc 7105 | Topic: comp.windows.x ===
Text preview: Does anyone have any information/advice on large color monitors
(17"-21") to use with a 486 system running X server software?
I maining looking for quality information and price, but all
information i
5 most similar:
  1. sim=0.2028 | ✗ comp.graphics
  2. sim=0.1915 | ✗ rec.autos
  3. sim=0.1884 | ✗ comp.graphics
  4. sim=0.1816 | ✗ sci.med
  5. sim=0.1808 | ✗ comp.sys.ibm.pc.hardware

=== Query doc 7764 | Topic: rec.motorcycles ===
Text preview: I may

## Pregunta 2

In [35]:
# --- From scratch: 1-NN prototype classifier ---
# cosine_similarity(X_test, X_train) → shape (N_test, N_train)
# For each test doc (row), find the argmax (most similar training doc)

from sklearn.metrics import f1_score, classification_report

batch_size = 500  # process in batches to avoid large dense matrices
y_pred_proto = np.zeros(len(y_test), dtype=int)

for start in range(0, len(y_test), batch_size):
    end = min(start + batch_size, len(y_test))
    batch_sims = cosine_similarity(X_test[start:end], X_train)  # (batch, N_train)
    nearest_idx = np.argmax(batch_sims, axis=1)                 # (batch,)
    y_pred_proto[start:end] = y_train[nearest_idx]

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"Prototype (1-NN) classifier F1-macro: {f1_proto:.4f}")

Prototype (1-NN) classifier F1-macro: 0.5050


In [36]:
# Per-class breakdown — which newsgroups does the prototype confuse?
print(classification_report(y_test, y_pred_proto, target_names=newsgroups_test.target_names))

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65      0.49      0.56       396
         

Clases como talk.politics.guns y talk.religion.misc comparten vocabulario político. El clasificador basado en prototipos tenderá a confundirlas con mayor frecuencia. Los grupos de noticias con temas más distintivos, por ejemplo rec.sport.hockey, suelen tener un F1 más alto.

## Pregunta 3

Se crea un baseline con MultinomialNB

In [37]:
# --- Baseline: MultinomialNB with default TF-IDF ---
clf_mnb = MultinomialNB(alpha=1.0)
clf_mnb.fit(X_train, y_train)
y_pred_mnb = clf_mnb.predict(X_test)
f1_mnb = f1_score(y_test, y_pred_mnb, average='macro')
print(f"MultinomialNB baseline F1-macro: {f1_mnb:.4f}")
# expected: ~0.67

MultinomialNB baseline F1-macro: 0.5854


Se crea un baseline con ComplementNB

In [38]:
# --- ComplementNB with default TF-IDF ---
clf_cnb = ComplementNB(alpha=1.0)
clf_cnb.fit(X_train, y_train)
y_pred_cnb = clf_cnb.predict(X_test)
f1_cnb = f1_score(y_test, y_pred_cnb, average='macro')
print(f"ComplementNB baseline F1-macro:  {f1_cnb:.4f}")
# expected: ~0.70-0.72 (better than MNB)

ComplementNB baseline F1-macro:  0.6930


Se prueba con algunas configuraciones de hiperparametros en un dict

* min_df=2: elimina palabras que aparecen solo en 1 documento, también llamadas hapax legomena. Esto reduce el ruido y el sobreajuste.
* sublinear_tf=True: reemplaza la frecuencia de término, TF, por 1 + log(TF). Esto evita que los documentos con muchas repeticiones de una sola palabra dominen el modelo.
* alpha=0.1: aplica menos suavizado de Laplace, por lo que el modelo confía más en las frecuencias observadas.
* ComplementNB: usa la distribución de las clases complementarias. Esto corrige el sesgo sistemático que suele tener MultinomialNB en problemas de clasificación de texto.

In [39]:
# --- Hyperparameter sweep ---
# Explore: sublinear_tf, min_df, alpha
# Do NOT change ngram_range per challenge instructions

from itertools import product

configs = [
    dict(sublinear_tf=False, min_df=1,  alpha=1.0,  model='MNB'),
    dict(sublinear_tf=True,  min_df=1,  alpha=1.0,  model='MNB'),
    dict(sublinear_tf=True,  min_df=2,  alpha=1.0,  model='MNB'),
    dict(sublinear_tf=True,  min_df=5,  alpha=0.1,  model='MNB'),
    dict(sublinear_tf=False, min_df=1,  alpha=1.0,  model='CNB'),
    dict(sublinear_tf=True,  min_df=1,  alpha=1.0,  model='CNB'),
    dict(sublinear_tf=True,  min_df=2,  alpha=1.0,  model='CNB'),
    dict(sublinear_tf=True,  min_df=2,  alpha=0.1,  model='CNB'),
    dict(sublinear_tf=True,  min_df=5,  alpha=0.1,  model='CNB'),
]

results = []
for cfg in configs:
    vect = TfidfVectorizer(sublinear_tf=cfg['sublinear_tf'], min_df=cfg['min_df'])
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)
    
    clf = (MultinomialNB if cfg['model'] == 'MNB' else ComplementNB)(alpha=cfg['alpha'])
    clf.fit(Xtr, y_train)
    f1 = f1_score(y_test, clf.predict(Xte), average='macro')
    results.append((f1, cfg))
    print(f"F1={f1:.4f} | {cfg}")

best_f1, best_cfg = sorted(results, reverse=True)[0]
print(f"\nBest F1-macro: {best_f1:.4f}")
print(f"Best config:   {best_cfg}")

F1=0.5854 | {'sublinear_tf': False, 'min_df': 1, 'alpha': 1.0, 'model': 'MNB'}
F1=0.5860 | {'sublinear_tf': True, 'min_df': 1, 'alpha': 1.0, 'model': 'MNB'}
F1=0.6016 | {'sublinear_tf': True, 'min_df': 2, 'alpha': 1.0, 'model': 'MNB'}
F1=0.6710 | {'sublinear_tf': True, 'min_df': 5, 'alpha': 0.1, 'model': 'MNB'}
F1=0.6930 | {'sublinear_tf': False, 'min_df': 1, 'alpha': 1.0, 'model': 'CNB'}
F1=0.6920 | {'sublinear_tf': True, 'min_df': 1, 'alpha': 1.0, 'model': 'CNB'}
F1=0.6932 | {'sublinear_tf': True, 'min_df': 2, 'alpha': 1.0, 'model': 'CNB'}
F1=0.6900 | {'sublinear_tf': True, 'min_df': 2, 'alpha': 0.1, 'model': 'CNB'}
F1=0.6771 | {'sublinear_tf': True, 'min_df': 5, 'alpha': 0.1, 'model': 'CNB'}

Best F1-macro: 0.6932
Best config:   {'sublinear_tf': True, 'min_df': 2, 'alpha': 1.0, 'model': 'CNB'}


In [40]:
# --- Train best model and show per-class report ---
best = best_cfg
vect_best = TfidfVectorizer(sublinear_tf=best['sublinear_tf'], min_df=best['min_df'])
Xtr_best = vect_best.fit_transform(newsgroups_train.data)
Xte_best = vect_best.transform(newsgroups_test.data)

clf_best = (MultinomialNB if best['model'] == 'MNB' else ComplementNB)(alpha=best['alpha'])
clf_best.fit(Xtr_best, y_train)
y_pred_best = clf_best.predict(Xte_best)

print(classification_report(y_test, y_pred_best, target_names=newsgroups_test.target_names))

                          precision    recall  f1-score   support

             alt.atheism       0.31      0.42      0.36       319
           comp.graphics       0.71      0.72      0.72       389
 comp.os.ms-windows.misc       0.73      0.57      0.64       394
comp.sys.ibm.pc.hardware       0.64      0.72      0.68       392
   comp.sys.mac.hardware       0.77      0.73      0.75       385
          comp.windows.x       0.80      0.80      0.80       395
            misc.forsale       0.76      0.75      0.76       390
               rec.autos       0.83      0.74      0.78       396
         rec.motorcycles       0.83      0.78      0.81       398
      rec.sport.baseball       0.92      0.84      0.88       397
        rec.sport.hockey       0.86      0.93      0.89       399
               sci.crypt       0.75      0.80      0.78       396
         sci.electronics       0.71      0.55      0.62       393
                 sci.med       0.80      0.81      0.80       396
         

Qué observar en el reporte:

* ¿Qué clases tienen F1 bajo? Esas son las clases donde el clasificador se está confundiendo, por ejemplo, pares con topicos similares como talk.religion.misc que se confunde con talk.politics.misc.
* sublinear_tf=True ayuda porque evita que las palabras muy frecuentes dentro de un documento dominen el puntaje.
* Un alpha más bajo, es decir, menos suavizado, suele ayudar a ComplementNB porque el vocabulario es lo suficientemente grande como para que los problemas por conteos cero sean poco frecuentes.

## Problema 4

In [49]:
# --- From scratch: transpose and compute word similarity ---
# Re-fit a fresh vectorizer
tfidf_ws = TfidfVectorizer()
X_train_ws = tfidf_ws.fit_transform(newsgroups_train.data)
idx2word_ws = {v: k for k, v in tfidf_ws.vocabulary_.items()}

# Transpose: shape becomes (V_words × N_docs)
X_term_doc = X_train_ws.T  # sparse matrix (V × N)
print(f"X_term_doc shape: {X_term_doc.shape}")

X_term_doc shape: (101631, 11314)


Se eligen manualmente ['hockey', 'gun', 'medical', 'computer', 'religion']

In [ ]:
words_of_interest = ['hockey', 'gun', 'medical', 'computer', 'religion']

# Verify they're in the vocabulary
for w in words_of_interest:
    if w in tfidf_ws.vocabulary_:
        print(f"'{w}' → index {tfidf_ws.vocabulary_[w]}")
    else:
        print(f"'{w}' NOT in vocabulary — choose another word")

'hockey' → index 47021
'gun' → index 44820
'medical' → index 60703
'computer' → index 28940
'religion' → index 77274


Excluimos la misma palabra (self)

In [46]:
# --- Compute top-5 similar words for each word of interest ---
print("\nWord Similarity Results:")
print("=" * 50)

for word in words_of_interest:
    if word not in tfidf_ws.vocabulary_:
        print(f"'{word}' not found, skipping")
        continue
    
    word_idx = tfidf_ws.vocabulary_[word]
    word_vec = X_term_doc[word_idx]        # shape: (1, N_train) sparse
    
    sims = cosine_similarity(word_vec, X_term_doc)[0]  # shape: (V,)
    
    top5_idx = np.argsort(sims)[::-1][1:6]  # exclude self
    similar_words = [(idx2word_ws[i], sims[i]) for i in top5_idx]
    
    print(f"\n'{word}' most similar words:")
    for similar_word, sim in similar_words:
        print(f"  {similar_word:<20} sim={sim:.4f}")


Word Similarity Results:

'hockey' most similar words:
  ncaa                 sim=0.2743
  nhl                  sim=0.2653
  affiliates           sim=0.2480
  xenophobes           sim=0.2426
  sportschannel        sim=0.2228

'gun' most similar words:
  guns                 sim=0.3582
  crime                sim=0.2441
  handgun              sim=0.2391
  homicides            sim=0.2331
  firearms             sim=0.2328

'medical' most similar words:
  romano               sim=0.2823
  hospitals            sim=0.2751
  recuperation         sim=0.2682
  providers            sim=0.2391
  relelvant            sim=0.2278

'computer' most similar words:
  decwriter            sim=0.1563
  deluged              sim=0.1522
  harkens              sim=0.1522
  shopper              sim=0.1443
  the                  sim=0.1361

'religion' most similar words:
  religious            sim=0.2451
  religions            sim=0.2116
  categorized          sim=0.2039
  purpsoe              sim=0.2008
  crus

Validamos que la palabra sobre si misma salga con similaridad igual 1

In [48]:
# --- Verification: self-similarity should be 1.0 ---
word = 'hockey'
word_idx = tfidf_ws.vocabulary_[word]
word_vec = X_term_doc[word_idx]
self_sim = cosine_similarity(word_vec, X_term_doc[word_idx])[0, 0]
print(f"Self-similarity of '{word}': {self_sim:.6f}")

Self-similarity of 'hockey': 1.000000


Hallazgos

* Las palabras del mismo tema o grupo de noticias tienden a agruparse.
* En el caso de gun y guns que tienen una similaridad morfologica, hubiera esperado una similtiud alta, sin embargo, tiene 0.36.
* Una similitud máxima muy baja, menor a 0.2, podria darse debido a que la palabra aparece en muy pocos documentos; por eso, su vector es disperso y ruidoso.

Esto es exactamente lo que aprende Word2Vec. Pero, en lugar de usar documentos completos como contexto, utiliza una ventana deslizante de 5 a 10 palabras alrededor de cada término. Esto produce vectores mucho más densos y semánticamente más precisos.